# dashboard_manifest_loader_v1

目标：验证 `src/dashboard` 可以优先读取 `analysis_manifest.json` 指向的统一 sidecar，而不是依赖散落实验固定文件名。

In [1]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

REPO = Path('/Users/rziiiii/dev/workspace')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.dashboard import build_dashboard_view_model, load_model_matrix_dashboard, write_model_matrix_dashboard_export

CARD = Path('research/dashboard_manifest_loader_v1')
MATRIX_CARD = CARD / 'matrix_card'
EXPORT_PATH = CARD / 'dashboard_export.json'
CARD.mkdir(parents=True, exist_ok=True)
MATRIX_CARD.mkdir(parents=True, exist_ok=True)
{
    'card': str(CARD),
    'matrix_card': str(MATRIX_CARD),
    'dashboard_export': str(EXPORT_PATH),
}


{'card': 'research/dashboard_manifest_loader_v1',
 'matrix_card': 'research/dashboard_manifest_loader_v1/matrix_card',
 'dashboard_export': 'research/dashboard_manifest_loader_v1/dashboard_export.json'}

## 构造 manifest-first matrix card

本 card 使用自定义 sidecar 文件名。固定文件名如 `analysis_summary.json`、`model_comparison.csv` 不存在，dashboard 只能通过 `analysis_manifest.json` 找到公开产物。

In [2]:
summary = {
    'schema_version': 'analysis.report.v1',
    'status': 'dashboard_manifest_loader_completed',
    'model_count': 2,
    'note': '中文 review：dashboard 通过 manifest 读取 sidecar。',
}
(MATRIX_CARD / 'manifest_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

pd.DataFrame([
    {'rank': 1, 'model': 'ridge_mse__top_ic_factors', 'base_model': 'ridge_mse', 'feature_set': 'top_ic_factors', 'return_pct': 4.2, 'relative_return_pct': 1.3, 'stability_score': 3.1},
    {'rank': 2, 'model': 'gcn_mse__all_features', 'base_model': 'gcn_mse', 'feature_set': 'all_features', 'return_pct': -1.1, 'relative_return_pct': -2.0, 'stability_score': -1.5},
]).to_csv(MATRIX_CARD / 'manifest_comparison.csv', index=False)
pd.DataFrame([
    {'model': 'ridge_mse__top_ic_factors', 'regime': 'bull', 'return_pct': 3.0},
    {'model': 'gcn_mse__all_features', 'regime': 'stress', 'return_pct': -4.0},
]).to_csv(MATRIX_CARD / 'manifest_regime.csv', index=False)
pd.DataFrame([
    {'model': 'ridge_mse__top_ic_factors', 'window': 'test', 'return_pct': 4.2},
    {'model': 'gcn_mse__all_features', 'window': 'test', 'return_pct': -1.1},
]).to_csv(MATRIX_CARD / 'manifest_window.csv', index=False)
pd.DataFrame([
    {'model': 'gcn_mse__all_features', 'window': 'test', 'regime': 'stress', 'segment': 'test::stress', 'return_pct': -4.0},
]).to_csv(MATRIX_CARD / 'manifest_window_regime.csv', index=False)
pd.DataFrame([
    {'model': 'ridge_mse__top_ic_factors', 'return_kind': 'strategy_return', 'annualized_return': 0.44},
    {'model': 'gcn_mse__all_features', 'return_kind': 'strategy_return', 'annualized_return': -0.18},
]).to_csv(MATRIX_CARD / 'manifest_risk.csv', index=False)
pd.DataFrame([
    {'model': 'gcn_mse__all_features', 'segment_type': 'window_regime', 'segment': 'test::stress', 'window': 'test', 'regime': 'stress', 'diagnosis': '弱于基准：压力窗口回撤扩大。'},
]).to_csv(MATRIX_CARD / 'manifest_deltas.csv', index=False)
pd.DataFrame([
    {'model': 'ridge_mse', 'paper_fidelity': 'native_library', 'promotion_blocker': ''},
    {'model': 'gcn_mse', 'paper_fidelity': 'proxy_feature_adapter', 'promotion_blocker': 'graph proxy'},
]).to_csv(MATRIX_CARD / 'manifest_capabilities.csv', index=False)
pd.DataFrame([{'model': 'ridge_mse__top_ic_factors', 'prediction': 0.1}]).to_csv(MATRIX_CARD / 'private_predictions.csv', index=False)

manifest = {
    'artifact_id': 'dashboard_manifest_loader_v1',
    'artifact_type': 'analysis_report',
    'producer': 'src.analysis.exports',
    'schema_version': 'artifact.v1',
    'paths': {
        'summary': 'manifest_summary.json',
        'model_comparison': 'manifest_comparison.csv',
        'regime_metrics': 'manifest_regime.csv',
        'window_metrics': 'manifest_window.csv',
        'window_regime_metrics': 'manifest_window_regime.csv',
        'qlib_risk_metrics': 'manifest_risk.csv',
        'model_effect_deltas': 'manifest_deltas.csv',
        'model_capabilities': 'manifest_capabilities.csv',
        'predictions': 'private_predictions.csv',
    },
    'metadata': {'analysis_report_schema_version': 'analysis.report.v1'},
    'manifest_hash': 'm' * 64,
}
(MATRIX_CARD / 'analysis_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
sorted(path.name for path in MATRIX_CARD.iterdir())


['analysis_manifest.json',
 'manifest_capabilities.csv',
 'manifest_comparison.csv',
 'manifest_deltas.csv',
 'manifest_regime.csv',
 'manifest_risk.csv',
 'manifest_summary.json',
 'manifest_window.csv',
 'manifest_window_regime.csv',
 'private_predictions.csv']

## 读取 dashboard payload

如果实现仍依赖固定文件名，这一步会在读取 summary 或 comparison 时失败。

In [3]:
payload = load_model_matrix_dashboard(MATRIX_CARD, top_n=2)
assert payload['artifact_id'] == 'dashboard_manifest_loader_v1'
assert payload['summary']['status'] == 'dashboard_manifest_loader_completed'
assert payload['top_models'][0]['model'] == 'ridge_mse__top_ic_factors'
assert payload['artifacts']['analysis_manifest'].endswith('analysis_manifest.json')
assert payload['artifacts']['model_comparison'].endswith('manifest_comparison.csv')
assert 'predictions' not in payload['artifacts']
{
    'artifact_id': payload['artifact_id'],
    'top_models': [row['model'] for row in payload['top_models']],
    'artifact_keys': sorted(payload['artifacts']),
    'risk_rows': len(payload['qlib_risk_metrics']),
    'delta_rows': len(payload['model_effect_deltas']),
}


{'artifact_id': 'dashboard_manifest_loader_v1',
 'top_models': ['ridge_mse__top_ic_factors', 'gcn_mse__all_features'],
 'artifact_keys': ['analysis_manifest',
  'model_capabilities',
  'model_comparison',
  'model_effect_deltas',
  'qlib_risk_metrics',
  'regime_metrics',
  'summary',
  'window_metrics',
  'window_regime_metrics'],
 'risk_rows': 2,
 'delta_rows': 1}

## 写出 dashboard export / view model

这里验证 manifest-first payload 仍可进入既有 dashboard export 和 view model 链路。

In [4]:
export_payload = write_model_matrix_dashboard_export(MATRIX_CARD, output_path=EXPORT_PATH, top_n=2)
view_model = build_dashboard_view_model(export_payload)
assert EXPORT_PATH.exists()
assert view_model['schema_version'] == 'dashboard.view_model.v1'
assert len(view_model['model_cards']) == 2
{
    'export_schema': export_payload['schema_version'],
    'view_model_schema': view_model['schema_version'],
    'model_card_count': len(view_model['model_cards']),
    'controls': view_model['controls'],
}


{'export_schema': 'dashboard.model_matrix.v1',
 'view_model_schema': 'dashboard.view_model.v1',
 'model_card_count': 2,
 'controls': {'models': ['gcn_mse__all_features', 'ridge_mse__top_ic_factors'],
  'base_models': ['gcn_mse', 'ridge_mse'],
  'feature_sets': ['all_features', 'top_ic_factors'],
  'regimes': ['bull', 'stress'],
  'windows': ['test'],
  'paper_fidelities': ['native_library', 'proxy_feature_adapter'],
  'attribution_features': []}}

## 写出本实验摘要

根目录摘要记录本轮 dashboard manifest-first 验证结果；dashboard export 保留为 sidecar。

In [5]:
run_summary = {
    'schema_version': 'research.dashboard_manifest_loader_v1',
    'status': 'completed',
    'matrix_card': str(MATRIX_CARD),
    'dashboard_export': str(EXPORT_PATH),
    'payload_schema': payload['schema_version'],
    'artifact_id': payload['artifact_id'],
    'artifact_keys': sorted(payload['artifacts']),
    'top_model': payload['top_models'][0]['model'],
    'model_card_count': len(view_model['model_cards']),
    'private_predictions_hidden': 'predictions' not in payload['artifacts'],
    'conclusion': 'src/dashboard 已支持优先读取 analysis_manifest.json 指向的公开 sidecar，且不会暴露 predictions。',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(run_summary, ensure_ascii=False, indent=2), encoding='utf-8')
run_summary


{'schema_version': 'research.dashboard_manifest_loader_v1',
 'status': 'completed',
 'matrix_card': 'research/dashboard_manifest_loader_v1/matrix_card',
 'dashboard_export': 'research/dashboard_manifest_loader_v1/dashboard_export.json',
 'payload_schema': 'dashboard.model_matrix.v1',
 'artifact_id': 'dashboard_manifest_loader_v1',
 'artifact_keys': ['analysis_manifest',
  'model_capabilities',
  'model_comparison',
  'model_effect_deltas',
  'qlib_risk_metrics',
  'regime_metrics',
  'summary',
  'window_metrics',
  'window_regime_metrics'],
 'top_model': 'ridge_mse__top_ic_factors',
 'model_card_count': 2,
 'private_predictions_hidden': True,
 'conclusion': 'src/dashboard 已支持优先读取 analysis_manifest.json 指向的公开 sidecar，且不会暴露 predictions。'}

## 中文结论

- 做了什么：验证 `src/dashboard` 的模型矩阵读取改为 manifest-first，可从 `analysis_manifest.json` 解析自定义 sidecar 文件名。
- 核心指标：本轮 dashboard payload 读取 2 个模型、2 行 qlib risk、1 行模型差异诊断，并成功生成 `dashboard_export.json` 和 view model。
- regime 差异：`manifest_regime.csv` 和 `manifest_window_regime.csv` 通过 manifest 进入 payload，压力窗口诊断保留在 `model_effect_deltas`。
- 是否晋升：这是框架能力，不是策略晋升；它减少 dashboard 对散落实验文件名的耦合。
- 风险：当前仍保留固定文件名 fallback 以兼容旧 card；后续应逐步让新 card 都写 `analysis_manifest.json`。
- 下一步：让 schedule/dashboard refresh 在 payload 中标记 manifest_source，并对旧 card 增加迁移/补 manifest 工具。